# 91 — Security Operations Supervisor

**Pattern:** Hierarchical supervisor with conditional subagent dispatch + materiality-gated escalation, built on LangGraph's `StateGraph` + `interrupt()`/`Command(resume=...)`.
**Key insight:** Running every specialist on every signal, every day, is noise. A real supervisor inspects the signal first and dispatches only the domains that have something to say — and escalates only the findings material enough to need a human decision before they're actionable.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/91-security-operations-supervisor/security_operations_workbook.ipynb)

In [ ]:
%pip install -q langgraph langchain-openai langchain-core pydantic python-dotenv

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## 1. Why "conditional" has to be real

A daily security feed has four independent signal sources: auth events, a CVE delta, an IAM diff, and open incident tickets. Most days, only one or two of those actually have something wrong in them. A supervisor that calls all five domain specialists every day regardless is not "hierarchical orchestration" — it's a fixed pipeline with extra steps.

The lesson here is the *dispatch check itself*: each domain subagent only runs if its slice of the digest has active signal (a flagged auth event, a non-empty IAM diff, at least one CVE, an open ticket). `compliance_evidence` is the exception — it always runs once anything else has, because its whole job is auditing what the others found.

## 2. The schema — signal digest in, typed findings + posture brief out

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field


class AuthEvent(BaseModel):
    identity: str
    event_type: str = Field(description="e.g. 'admin_login', 'impossible_travel', 'off_hours_login'")
    detail: str
    flagged: bool = Field(description="True if the SIEM flagged this event as anomalous")


class CVEDelta(BaseModel):
    package: str
    cve_id: str
    severity: Literal["low", "medium", "high", "critical"]
    actively_exploited: bool


class IAMChange(BaseModel):
    identity: str
    change_type: Literal["new_admin_grant", "orphaned_account"]
    role: str
    detail: str


class IncidentTicket(BaseModel):
    ticket_id: str
    description: str
    severity: Literal["low", "medium", "high", "critical"]
    status: str


class SecuritySignalDigest(BaseModel):
    auth_events: list[AuthEvent] = Field(default_factory=list)
    cve_deltas: list[CVEDelta] = Field(default_factory=list)
    iam_changes: list[IAMChange] = Field(default_factory=list)
    open_incident: Optional[IncidentTicket] = None


class SecurityFindingReport(BaseModel):
    domain: Literal[
        "threat_triage", "access_governance", "vulnerability_remediation",
        "incident_commander", "compliance_evidence",
    ]
    severity: Literal["P0", "P1", "P2", "P3"]
    summary: str
    recommended_action: str
    requires_approval: bool = Field(
        description="True if P0/P1 -- must clear the approval gate before it's actionable"
    )


class SecurityPostureBrief(BaseModel):
    dispatched_domains: list[str]
    findings: list[SecurityFindingReport]
    cross_domain_correlations: list[str] = Field(default_factory=list)
    overall_posture: Literal["green", "amber", "red"]


class ProposedAction(BaseModel):
    action_type: Literal["revoke_access", "apply_patch", "send_notice"]
    summary: str
    payload: dict
    risk_level: Literal["low", "medium", "high"]


class ApprovalDecision(BaseModel):
    decision: Literal["approve", "edit", "reject"]
    edited_payload: Optional[dict] = None
    rationale: str


class ActionResult(BaseModel):
    executed: bool
    final_payload: Optional[dict] = None
    decision_log: str

## 3. Five domain subagents, one shared shape

Each subagent gets only its slice of the digest and returns a `SecurityFindingReport`. Notice `_run()` forces both `domain` and `requires_approval` after the LLM call instead of trusting the model to set them — `domain` is always known at the call site (whichever agent you just called), and `requires_approval` is nothing more than "is severity P0 or P1", so it's computed from the model's own severity call rather than asked for separately. In practice the model doesn't reliably follow either instruction on its own: `compliance_evidence` sometimes echoed another domain's label back, and could just as easily mark a genuine P0 as not needing approval in any of the other four domains — which would let a real revoke-access or apply-patch recommendation skip the gate entirely. When a value is fully known (or fully derivable) at the call site, don't leave it to the model.

In [ ]:
import json
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI

_MODEL = "gpt-4.1-nano"

THREAT_TRIAGE_SYSTEM = SystemMessage(
    "You are a threat-triage analyst. You receive a list of SIEM auth events, at least one of "
    "which is flagged anomalous. Summarise the pattern across the flagged events (e.g. "
    "impossible-travel, off-hours admin logins), assign a severity (P0 for active compromise "
    "indicators, P1 for high-confidence anomalies, P2/P3 for lower-confidence signal), and "
    "recommend a concrete triage action. Set requires_approval=true only for P0/P1. domain must "
    "be 'threat_triage'."
)
ACCESS_GOVERNANCE_SYSTEM = SystemMessage(
    "You are an access-governance analyst. You receive a list of IAM changes: new admin grants "
    "and orphaned accounts still holding active privileges. Summarise the riskiest change, assign "
    "a severity (P0 for an orphaned account with standing privileged access, P1 for an "
    "unexplained new admin grant, P2/P3 otherwise), and recommend a concrete remediation (e.g. "
    "revoke access). Set requires_approval=true only for P0/P1. domain must be 'access_governance'."
)
VULNERABILITY_REMEDIATION_SYSTEM = SystemMessage(
    "You are a vulnerability-remediation analyst. You receive a list of new/changed CVEs from a "
    "dependency scan. Summarise the most urgent CVE, assign a severity (P0 if any CVE is actively "
    "exploited, P1 for critical/high severity otherwise, P2/P3 for medium/low), and recommend a "
    "concrete patch action. Set requires_approval=true only for P0/P1. domain must be "
    "'vulnerability_remediation'."
)
INCIDENT_COMMANDER_SYSTEM = SystemMessage(
    "You are an incident commander. You receive one open incident ticket. Summarise its blast "
    "radius, assign a severity matching or escalating the ticket's stated severity, and recommend "
    "a concrete next containment or communication action. Set requires_approval=true only for "
    "P0/P1. domain must be 'incident_commander'."
)
COMPLIANCE_EVIDENCE_SYSTEM = SystemMessage(
    "You are a compliance-evidence analyst. You receive the typed findings already produced by "
    "the other security subagents today. Identify whether any finding represents a control gap "
    "that would need fresh evidence for a SOC2/ISO27001 audit (e.g. an unremediated P0/P1 with no "
    "logged action), summarise the single most material gap, assign it a severity, and recommend "
    "collecting the missing evidence artifact. Set requires_approval=false always -- this domain "
    "surfaces gaps, it does not propose irreversible actions. domain must be 'compliance_evidence'."
)


def _run(llm, system, payload, domain, always_advisory=False):
    # domain and requires_approval are both fully known at the call site, so
    # force them deterministically rather than trust the model to set them --
    # it sometimes echoes another domain's label back, and doesn't reliably
    # follow "requires_approval=true only for P0/P1" either.
    structured = llm.with_structured_output(SecurityFindingReport, method="function_calling")
    result = structured.invoke([system, ("human", json.dumps(payload))])
    requires_approval = False if always_advisory else result.severity in ("P0", "P1")
    return result.model_copy(update={"domain": domain, "requires_approval": requires_approval})


def threat_triage_agent(llm, auth_events):
    flagged = [e.model_dump() for e in auth_events if e.flagged]
    return _run(llm, THREAT_TRIAGE_SYSTEM, flagged, "threat_triage")

def access_governance_agent(llm, iam_changes):
    return _run(llm, ACCESS_GOVERNANCE_SYSTEM, [c.model_dump() for c in iam_changes], "access_governance")

def vulnerability_remediation_agent(llm, cve_deltas):
    return _run(llm, VULNERABILITY_REMEDIATION_SYSTEM, [d.model_dump() for d in cve_deltas], "vulnerability_remediation")

def incident_commander_agent(llm, incident):
    return _run(llm, INCIDENT_COMMANDER_SYSTEM, incident.model_dump(), "incident_commander")

def compliance_evidence_agent(llm, findings):
    return _run(llm, COMPLIANCE_EVIDENCE_SYSTEM, [f.model_dump() for f in findings],
                "compliance_evidence", always_advisory=True)

## 4. The graph — dispatch_domains → [human_review → resolve_gate] → synthesize

`dispatch_domains` conditionally calls only the relevant agents and collects their findings. A conditional edge routes to `human_review` — a real `interrupt()` — only if at least one finding is P0/P1; otherwise straight to `synthesize`. `compliance_evidence`'s self-assigned severity is excluded from the overall posture calculation, since it reports on evidence gaps, not live operational risk.

In [ ]:
import uuid
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

_ACTION_TYPE_BY_DOMAIN = {
    "threat_triage": "revoke_access",
    "access_governance": "revoke_access",
    "vulnerability_remediation": "apply_patch",
    "incident_commander": "send_notice",
    "compliance_evidence": "send_notice",
}
_SEVERITY_RANK = {"P0": 0, "P1": 1, "P2": 2, "P3": 3}


class GraphState(TypedDict):
    digest: dict
    dispatched_domains: list
    findings: list
    gated_finding: dict
    decision: dict
    gate_result: dict
    brief: dict


def _detect_correlations(digest: SecuritySignalDigest) -> list[str]:
    orphaned = {c.identity for c in digest.iam_changes if c.change_type == "orphaned_account"}
    flagged = {e.identity for e in digest.auth_events if e.flagged}
    overlap = orphaned & flagged
    return [
        f"Identity {identity!r} is both an orphaned-but-privileged account and the actor in a "
        "flagged anomalous auth event -- investigate as one incident, not two."
        for identity in sorted(overlap)
    ]


def _dispatch_domains(state):
    digest = SecuritySignalDigest.model_validate(state["digest"])
    llm = ChatOpenAI(model=_MODEL, temperature=0)
    dispatched, findings = [], []

    if any(e.flagged for e in digest.auth_events):
        dispatched.append("threat_triage")
        findings.append(threat_triage_agent(llm, digest.auth_events))
    if digest.iam_changes:
        dispatched.append("access_governance")
        findings.append(access_governance_agent(llm, digest.iam_changes))
    if digest.cve_deltas:
        dispatched.append("vulnerability_remediation")
        findings.append(vulnerability_remediation_agent(llm, digest.cve_deltas))
    if digest.open_incident is not None:
        dispatched.append("incident_commander")
        findings.append(incident_commander_agent(llm, digest.open_incident))
    if findings:
        dispatched.append("compliance_evidence")
        findings.append(compliance_evidence_agent(llm, findings))

    return {"dispatched_domains": dispatched, "findings": [f.model_dump() for f in findings]}


def _route_after_dispatch(state):
    findings = [SecurityFindingReport.model_validate(f) for f in state["findings"]]
    return "human_review" if any(f.requires_approval for f in findings) else "synthesize"


def _human_review(state):
    findings = [SecurityFindingReport.model_validate(f) for f in state["findings"]]
    gated = sorted((f for f in findings if f.requires_approval), key=lambda f: _SEVERITY_RANK.get(f.severity, 9))
    top = gated[0]
    proposed = ProposedAction(
        action_type=_ACTION_TYPE_BY_DOMAIN.get(top.domain, "send_notice"),
        summary=top.recommended_action,
        payload={"domain": top.domain, "severity": top.severity, "summary": top.summary},
        risk_level="high" if top.severity == "P0" else "medium",
    )
    decision = interrupt({"proposed_action": proposed.model_dump(), "finding": top.model_dump()})
    return {"decision": decision, "gated_finding": top.model_dump()}


def _resolve_gate(state):
    top = SecurityFindingReport.model_validate(state["gated_finding"])
    decision = ApprovalDecision.model_validate(state["decision"])
    if decision.decision == "reject":
        result = ActionResult(executed=False, final_payload=None,
                               decision_log=f"REJECTED: {top.recommended_action!r}. Reason: {decision.rationale}")
    else:
        payload = decision.edited_payload if decision.decision == "edit" and decision.edited_payload else \
            {"domain": top.domain, "severity": top.severity, "summary": top.summary}
        verb = "EDITED-THEN-EXECUTED" if decision.decision == "edit" else "EXECUTED"
        result = ActionResult(executed=True, final_payload=payload,
                               decision_log=f"{verb}: {top.recommended_action!r}. Reason: {decision.rationale}")
    return {"gate_result": result.model_dump()}


def _synthesize(state):
    digest = SecuritySignalDigest.model_validate(state["digest"])
    findings = [SecurityFindingReport.model_validate(f) for f in state["findings"]]
    correlations = _detect_correlations(digest)
    operational = [f for f in findings if f.domain != "compliance_evidence"]
    if any(f.severity == "P0" for f in operational):
        posture = "red"
    elif any(f.severity in ("P1", "P2") for f in operational):
        posture = "amber"
    else:
        posture = "green"
    brief = SecurityPostureBrief(dispatched_domains=state["dispatched_domains"], findings=findings,
                                  cross_domain_correlations=correlations, overall_posture=posture)
    return {"brief": brief.model_dump()}


def _build_graph():
    graph = StateGraph(GraphState)
    graph.add_node("dispatch_domains", _dispatch_domains)
    graph.add_node("human_review", _human_review)
    graph.add_node("resolve_gate", _resolve_gate)
    graph.add_node("synthesize", _synthesize)
    graph.add_edge(START, "dispatch_domains")
    graph.add_conditional_edges("dispatch_domains", _route_after_dispatch,
                                 {"human_review": "human_review", "synthesize": "synthesize"})
    graph.add_edge("human_review", "resolve_gate")
    graph.add_edge("resolve_gate", "synthesize")
    graph.add_edge("synthesize", END)
    return graph.compile(checkpointer=InMemorySaver())


_APP = _build_graph()


def propose(digest: SecuritySignalDigest) -> dict:
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    result = _APP.invoke({"digest": digest.model_dump()}, config=config)
    if "__interrupt__" in result:
        value = result["__interrupt__"][0].value
        return {"status": "paused", "proposed_action": ProposedAction.model_validate(value["proposed_action"]),
                "finding": SecurityFindingReport.model_validate(value["finding"]), "thread_id": thread_id}
    return {"status": "complete", "brief": SecurityPostureBrief.model_validate(result["brief"])}


def resume(thread_id: str, decision: ApprovalDecision) -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    result = _APP.invoke(Command(resume=decision.model_dump()), config=config)
    return {"brief": SecurityPostureBrief.model_validate(result["brief"]),
            "gate_result": ActionResult.model_validate(result["gate_result"])}

## 5. Run it — a busy day (gated) vs a quiet day (no gate)

The busy day has a deliberate cross-domain correlation: the orphaned service account (`svc-batch-07`) is the same identity behind a flagged off-hours login. Watch it surface in `cross_domain_correlations` instead of as two disconnected findings.

In [ ]:
busy_digest = SecuritySignalDigest(
    auth_events=[
        AuthEvent(identity="svc-batch-07", event_type="off_hours_login",
                  detail="03:14 UTC from an unrecognised ASN", flagged=True),
        AuthEvent(identity="jdoe", event_type="impossible_travel",
                  detail="Toronto 09:02 UTC, then Singapore 09:41 UTC", flagged=True),
        AuthEvent(identity="user-001", event_type="standard_login",
                  detail="09:01 local time, corporate VPN range", flagged=False),
    ],
    cve_deltas=[
        CVEDelta(package="libxml2", cve_id="CVE-2026-11042", severity="critical", actively_exploited=True),
    ],
    iam_changes=[
        IAMChange(identity="svc-batch-07", change_type="orphaned_account", role="prod-write",
                  detail="96 days since last legitimate scheduled run"),
    ],
    open_incident=None,
)

result = propose(busy_digest)
print("Status:", result["status"])
if result["status"] == "paused":
    print("Gated finding:", result["finding"].domain, result["finding"].severity)
    print("Proposed action:", result["proposed_action"])

    decision = ApprovalDecision(decision="approve", rationale="Confirmed compromised, revoking now.")
    resumed = resume(result["thread_id"], decision)
    print("\nGate result:", resumed["gate_result"].decision_log)
    brief = resumed["brief"]
else:
    brief = result["brief"]

print("\nPosture:", brief.overall_posture)
print("Correlations:", brief.cross_domain_correlations)
for f in brief.findings:
    print(f"  [{f.severity}] {f.domain}: {f.summary}")

In [ ]:
quiet_digest = SecuritySignalDigest(
    auth_events=[],
    cve_deltas=[CVEDelta(package="pillow", cve_id="CVE-2026-10555", severity="medium", actively_exploited=False)],
    iam_changes=[],
    open_incident=None,
)

result = propose(quiet_digest)
print("Status:", result["status"], "-- only vulnerability_remediation + compliance_evidence should dispatch")
brief = result["brief"]
print("Dispatched:", brief.dispatched_domains)
print("Posture:", brief.overall_posture)

## 6. Starter Exercise

`_human_review` currently escalates only the **single** highest-severity gated finding, even if the digest produced more than one P0. Write `gate_candidates(findings: list[SecurityFindingReport]) -> list[SecurityFindingReport]` that returns *every* finding with `requires_approval=True`, sorted by severity (P0 first). Run it against `busy_digest`'s findings (before the gate collapses them to one) to see how many approvals a real system would actually need to queue.

In [ ]:
# Your code here

### Answer Key

In [ ]:
def gate_candidates(findings: list[SecurityFindingReport]) -> list[SecurityFindingReport]:
    gated = [f for f in findings if f.requires_approval]
    return sorted(gated, key=lambda f: _SEVERITY_RANK.get(f.severity, 9))


# Re-run dispatch alone (skip the gate) to inspect all raw findings pre-collapse.
llm = ChatOpenAI(model=_MODEL, temperature=0)
raw_findings = [
    threat_triage_agent(llm, busy_digest.auth_events),
    access_governance_agent(llm, busy_digest.iam_changes),
    vulnerability_remediation_agent(llm, busy_digest.cve_deltas),
]

candidates = gate_candidates(raw_findings)
print(f"{len(candidates)} finding(s) actually need an approval gate, not just the top one:")
for f in candidates:
    print(f"  [{f.severity}] {f.domain}: {f.summary}")